# La escalera de ablación — qué parte de la formulación hace qué

Dieciséis condiciones, cada una agregando **una** cosa sobre la anterior, en
limpio y contaminado, sobre dos transferencias.

| # | brazo | atención | copia | loss | qué aísla |
|---|---|---|---|---|---|
| 1 | B | β=1/m | — | — | **el piso verdadero** |
| 2a | B | nuestra, τ buscada | — | — | la atención en el régimen que la búsqueda eligió |
| 2b | B | nuestra, τ=0,5 | — | — | la atención discriminando |
| 2c | B | nuestra, τ=0,1 | — | — | la atención discriminando fuerte |
| 3a | E | nuestra, τ elegida | nunca | off | ¿alcanzan las estadísticas solas? |
| 3b | E | nuestra, τ elegida | N=7 | off | **copiar los pesos de fuente** |
| 3c | E | nuestra, τ elegida | N=1 | off | **copiar más seguido** |
| 3u | E | **β=1/m** | N=7 | off | **¿la copia necesita nuestra atención?** |
| 4a | E | nuestra, τ elegida | nunca | on | el Gram de bolsas, sin copia |
| 4b | E | nuestra, τ elegida | N=7 | on | **el Gram de bolsas** |
| 4c | E | nuestra, τ elegida | N=1 | on | el Gram, a copia máxima |
| 5a | F | nuestra, τ elegida | nunca | on | el sopesado, sin copia |
| 5b | F | nuestra, τ elegida | N=7 | on | **el sopesado por confianza** |
| 5c | F | nuestra, τ elegida | N=1 | on | el sopesado, a copia máxima |
| 6a | G | nuestra, τ elegida | nunca | on | el local, sin copia |
| 6b | G | nuestra, τ elegida | N=7 | on | **la correspondencia local** |
| 6c | G | nuestra, τ elegida | N=1 | on | el local, a copia máxima |

> **La temperatura de las filas 3 a 6 se elige sobre `valid` y nunca sobre lo
> que se reporta.** A la temperatura que la búsqueda eligió, β se separa de
> uniforme apenas un 7 a 9% (medido tras entrenar); a 0,5 se separa 36-41% y a
> 0,1 un 175-188%. Medir doce filas en un régimen donde la atención casi no
> actúa no tiene sentido, así que la escalera corre a la mejor de las tres ---
> pero elegirla mirando `eval` sería poner en la tabla el valor que más
> favorece al método, elegido por el resultado. Es el mismo cuidado que
> `config.NOISE_REPORTED` ya documenta para el nivel de ruido reportado, y la
> razón por la que `config.SEARCH_ROLE` es `valid`: la búsqueda no ve el número
> que publica. La elección se imprime ANTES de que corra el resto.

> **Las filas 2a, 2b y 2c sí se reportan sobre `eval`**, a sus tres
> temperaturas fijas: son la comparación de atención, y ninguna de ellas eligió
> su propio número.

> **El coeficiente del término queda FIJO**, el que la búsqueda eligió por
> transfer, y no se busca por fila: si cada fila buscara el suyo, la 5
> diferiría de la 4 en el sopesado Y en el coeficiente, y ninguna diferencia
> sería atribuible. El precio: esta rejilla no distingue «el componente no
> ayuda» de «no ayuda a ESTE coeficiente».

> **Lo que este cuaderno NO es.** No es un producto declarado del flujo: no
> escribe bajo `config.results_for(...)` ni lo lee ningún paso de `steps.py`.
> Convertirlo en un peldaño declarado es un acto de deliberación que nadie hizo.

In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [2]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

repository: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation


In [3]:
# `tau`: None significa "la elegida sobre valid"; un número es una temperatura
# fija y declarada. `copia` es el intervalo de transferencia, None = NUNCA, y
# el cuaderno lo AFIRMA contando las transferencias que de verdad ocurrieron.
#   id   brazo  atencion   copia  loss   tau
CONDICIONES = [
    ("1",  "B",  "uniform", None,  False, None),
    ("2a", "B",  "learned", None,  False, "buscada"),
    ("2b", "B",  "learned", None,  False, 0.5),
    ("2c", "B",  "learned", None,  False, 0.1),
    ("3a", "E",  "learned", None,  False, None),
    ("3b", "E",  "learned", 7,     False, None),
    ("3c", "E",  "learned", 1,     False, None),
    # El otro lado del cuadrado que cierra "son esos batch" contra "son esos
    # batch GRACIAS a nuestra metodologia". La fila 1 es uniforme SIN copia y
    # la 3b es nuestra atencion CON copia: entre esas dos cambian dos cosas a
    # la vez, asi que ninguna de las dos frases es escribible. Esta es
    # uniforme CON copia, y es la unica celda que las separa.
    ("3u", "E",  "uniform", 7,     False, None),
    ("4a", "E",  "learned", None,  True,  None),
    ("4b", "E",  "learned", 7,     True,  None),
    ("4c", "E",  "learned", 1,     True,  None),
    ("5a", "F",  "learned", None,  True,  None),
    ("5b", "F",  "learned", 7,     True,  None),
    ("5c", "F",  "learned", 1,     True,  None),
    ("6a", "G",  "learned", None,  True,  None),
    ("6b", "G",  "learned", 7,     True,  None),
    ("6c", "G",  "learned", 1,     True,  None),
]

#: Las tres candidatas de la elección sobre `valid`. "buscada" es la que el
#: registro de la búsqueda ya eligió para esa transferencia.
CANDIDATAS = ["buscada", 0.5, 0.1]

TRANSFERS = [("U", "M"), ("M", "S")]
SEMILLAS = [0, 1, 2, 3, 4]
EPOCAS = 12
TASAS = [0.0, 0.4]      # limpio, y concentrado a 18 propias contra 12

assert len({c[0] for c in CONDICIONES}) == len(CONDICIONES), "ids repetidos"
corridas = len(CONDICIONES) * len(TASAS) * len(TRANSFERS) * len(SEMILLAS)
print(f"{len(CONDICIONES)} condiciones x {len(TASAS)} tasas x "
      f"{len(TRANSFERS)} transfers x {len(SEMILLAS)} semillas = {corridas} corridas")
print(f"mas la eleccion sobre valid: {len(CANDIDATAS)} x {len(TRANSFERS)} = "
      f"{len(CANDIDATAS)*len(TRANSFERS)} corridas")

17 condiciones x 2 tasas x 2 transfers x 5 semillas = 340 corridas
mas la eleccion sobre valid: 3 x 2 = 6 corridas


## La elección de la temperatura, sobre `valid`

In [4]:
import statistics as st
import time

import torch

from MIL_CREDA_Benchmark import bags, config, harness, wiring

DISPOSITIVO = harness.resolve_device()
print("dispositivo:", DISPOSITIVO)

_RAMPA = harness.ramp
_SYNC = wiring.Arm.sync_target_from_source


def _cero(*a, **k):
    return 0.0


_material = {}


def material(code, semilla, tasa):
    clave = (code, semilla, tasa)
    if clave not in _material:
        _material[clave] = bags.build(code, config.DATA_CACHE, semilla, tasa)
    return _material[clave]


def reduccion_de(tasa):
    """`with_ceilings_in_force` inmediatamente antes de correr, nunca una
    `Reduction` armada a mano: el harness se niega a eso y con razón."""
    r = harness.with_ceilings_in_force(
        harness.Reduction(labelNoise=tasa, kind="probe", pilot=True,
                          epochs=EPOCAS, seeds=SEMILLAS), DISPOSITIVO)
    assert r.epochs == EPOCAS and r.seeds == SEMILLAS, "la escala no quedó"
    return r


def correr(brazo, atencion, copia, loss, tau, transfer, semilla, red, rol):
    """Una corrida, conducida por `harness.run_one` y nunca por un bucle propio.

    El bucle del harness resuelve el sorteo balanceado de bolsas, el cronograma
    de tasa de aprendizaje y la transferencia. Un bucle escrito acá se equivoca
    en el sorteo y muere dentro de `total_correspondence` cuando una clase no
    tiene bolsa de fuente.

    Devuelve la exactitud y CUÁNTAS transferencias ocurrieron, que es lo que
    vuelve comprobable la columna `copia` en vez de supuesta.
    """
    original = dict(config.ARMS_BY_ID[brazo])
    spec = dict(original)
    spec["attention"] = atencion
    hyper = dict(harness.hyper_for(red, original["adaptation"], transfer))
    if tau is not None and tau != "buscada":
        hyper["attentionTemperature"] = tau
    cuenta = {"n": 0}

    def contado(self):
        cuenta["n"] += 1
        return _SYNC(self)

    config.ARMS_BY_ID[brazo] = spec
    harness.ramp = _RAMPA if loss else _cero
    wiring.Arm.sync_target_from_source = (
        (lambda self: None) if copia is None else contado)
    previo = config.SOURCE_TO_TARGET_SYNC_STEPS
    if copia is not None:
        config.SOURCE_TO_TARGET_SYNC_STEPS = copia
    try:
        salida = harness.run_one(
            brazo, transfer, seed=semilla, device=DISPOSITIVO, role=rol,
            material={"source": material(transfer[0], semilla, red.labelNoise),
                      "target": material(transfer[1], semilla, red.labelNoise)},
            reduction=red, hyper=hyper)
    finally:
        config.ARMS_BY_ID[brazo] = original
        harness.ramp = _RAMPA
        wiring.Arm.sync_target_from_source = _SYNC
        config.SOURCE_TO_TARGET_SYNC_STEPS = previo
    return salida["targetAccuracy"], cuenta["n"]


# La elección: sobre `valid`, material limpio y una semilla, igual que la
# búsqueda declarada (`config.SEARCH_ROLE`). Se imprime ANTES de que corra lo
# que se reporta, para que no haya duda de que no vio `eval`.
print(f"\neligiendo la temperatura sobre rol '{config.SEARCH_ROLE}', "
      f"material limpio, semilla {SEMILLAS[0]}\n")
ELEGIDA = {}
red_limpia = reduccion_de(0.0)
for origen, destino in TRANSFERS:
    etiqueta = f"{origen}->{destino}"
    buscada = harness.hyper_for(
        red_limpia, "milcreda", (origen, destino))["attentionTemperature"]
    puntajes = {}
    for cand in CANDIDATAS:
        v, _ = correr("B", "learned", None, False, cand, (origen, destino),
                      SEMILLAS[0], red_limpia, config.SEARCH_ROLE)
        puntajes[cand] = v
        valor = buscada if cand == "buscada" else cand
        print(f"  {etiqueta}  tau={valor:<8.4g} ({cand})  "
              f"{config.SEARCH_ROLE}={v:.4f}")
    ELEGIDA[etiqueta] = max(puntajes, key=puntajes.get)
    valor = buscada if ELEGIDA[etiqueta] == "buscada" else ELEGIDA[etiqueta]
    print(f"  -> {etiqueta} corre las filas 3-6 a tau={valor:.4g} "
          f"({ELEGIDA[etiqueta]})\n")

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dispositivo: mps

eligiendo la temperatura sobre rol 'valid', material limpio, semilla 0



  U->M  tau=2.355    (buscada)  valid=0.6000


  U->M  tau=0.5      (0.5)  valid=0.4500


  U->M  tau=0.1      (0.1)  valid=0.4000
  -> U->M corre las filas 3-6 a tau=2.355 (buscada)



  M->S  tau=3.89     (buscada)  valid=0.1500


  M->S  tau=0.5      (0.5)  valid=0.2000


  M->S  tau=0.1      (0.1)  valid=0.1000
  -> M->S corre las filas 3-6 a tau=0.5 (0.5)



## La corrida

In [5]:
import json

SALIDA = REPOSITORY / "MIL-CREDA" / "Notebooks" / "ablation_ladder.json"

# Reanudable a proposito: una condicion agregada despues no obliga a repetir
# las 320 corridas que ya se midieron. La clave es (transfer, tasa, id), y una
# fila ya presente se saltea en vez de recomputarse. Si cambia la escala, el
# archivo viejo deja de servir y se descarta entero -- comparar filas de 12
# epocas con filas de otra escala seria peor que no tenerlas.
filas = []
if SALIDA.is_file():
    previo = json.loads(SALIDA.read_text(encoding="utf-8"))
    misma_escala = (previo.get("epocas") == EPOCAS
                    and previo.get("semillas") == SEMILLAS
                    and previo.get("tasas") == TASAS)
    if misma_escala:
        filas = previo.get("filas", [])
        print(f"reanudando: {len(filas)} filas ya medidas a la misma escala")
    else:
        print(f"el archivo previo corrio a otra escala "
              f"({previo.get('epocas')} epocas, {previo.get('semillas')} semillas) "
              f"-> se descarta entero")
ya = {(f["transfer"], f["tasa"], f["id"]) for f in filas}
t0 = time.time()
pendientes = sum(1 for tasa in TASAS for o, d in TRANSFERS
                 for c in CONDICIONES
                 if (f"{o}->{d}", tasa, c[0]) not in ya)
total = pendientes * len(SEMILLAS)
hecho = 0
print(f"condiciones pendientes: {pendientes}  -> {total} corridas")
pasos = -(-config.TRAIN_BAGS // config.BAGS_PER_STEP) * EPOCAS
ESPERADO = {None: 0, 7: pasos // 7, 1: pasos}
print(f"pasos de optimizador por corrida: {pasos}")
print(f"transferencias esperadas: {ESPERADO}\n")

for tasa in TASAS:
    red = reduccion_de(tasa)
    for origen, destino in TRANSFERS:
        etiqueta = f"{origen}->{destino}"
        for ident, brazo, atencion, copia, loss, tau in CONDICIONES:
            # None = la elegida sobre valid; un numero o "buscada" es declarado.
            if (etiqueta, tasa, ident) in ya:
                continue
            usada = (None if atencion != "learned"
                     else ELEGIDA[etiqueta] if tau is None else tau)
            valores, copias = [], set()
            for semilla in SEMILLAS:
                v, n = correr(brazo, atencion, copia, loss, usada,
                              (origen, destino), semilla, red, "eval")
                valores.append(v)
                copias.add(n)
                hecho += 1
            esperado = ESPERADO[copia] if brazo != "B" else 0
            assert copias == {esperado}, (
                f"{ident} declara copia={copia} y transfirió {sorted(copias)}, "
                f"no {esperado}")
            filas.append({
                "id": ident, "arm": brazo, "attention": atencion,
                "copia": copia, "loss": loss, "tau": usada,
                "tauElegida": tau is None, "transfer": etiqueta, "tasa": tasa,
                "media": st.mean(valores),
                "std": st.stdev(valores) if len(valores) > 1 else 0.0,
                "transferencias": esperado, "vals": valores})
            # Escrito DENTRO del bucle y no al final: `nbconvert` no muestra la
            # salida de una celda hasta que cierra, asi que el archivo es la
            # unica manera de ver el avance de una corrida de horas.
            SALIDA.write_text(json.dumps(
                {"tasas": TASAS, "epocas": EPOCAS, "semillas": SEMILLAS,
                 "elegida": ELEGIDA, "completo": hecho == total,
                 "filas": filas}, indent=1), encoding="utf-8")
            print(f"[{hecho:3}/{total}] {time.time()-t0:6.0f}s  rho={tasa:<4} "
                  f"{etiqueta:>5}  {ident:>3} {brazo} {atencion:>7} "
                  f"copia={str(copia):>4} loss={'on ' if loss else 'off'}  "
                  f"{st.mean(valores):.4f} +- "
                  f"{st.stdev(valores) if len(valores)>1 else 0:.4f}", flush=True)

print(f"\nescrito: {SALIDA}")
print(f"tiempo total: {time.time()-t0:.0f}s")

reanudando: 64 filas ya medidas a la misma escala
condiciones pendientes: 4  -> 20 corridas
pasos de optimizador por corrida: 84
transferencias esperadas: {None: 0, 7: 12, 1: 84}



[  5/20]    165s  rho=0.0   U->M   3u E uniform copia=   7 loss=off  0.9944 +- 0.0124


[ 10/20]    332s  rho=0.0   M->S   3u E uniform copia=   7 loss=off  0.5833 +- 0.1457


[ 15/20]    516s  rho=0.4   U->M   3u E uniform copia=   7 loss=off  0.8222 +- 0.0847


[ 20/20]    689s  rho=0.4   M->S   3u E uniform copia=   7 loss=off  0.4722 +- 0.0761



escrito: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/MIL-CREDA/Notebooks/ablation_ladder.json
tiempo total: 689s


## La tabla

In [6]:
por_clave = {(f["transfer"], f["tasa"], f["id"]): f for f in filas}
for origen, destino in TRANSFERS:
    etiqueta = f"{origen}->{destino}"
    valor = ELEGIDA[etiqueta]
    print(f"\n==== {etiqueta}   (filas 3-6 a tau {valor}) ====")
    print(f"{'#':>3} {'brazo':>6} {'atencion':>9} {'copia':>6} {'loss':>5} "
          f"{'tau':>9} {'ACC sin ruido':>16} {'ACC con ruido':>16}")
    for ident, brazo, atencion, copia, loss, tau in CONDICIONES:
        def celda(tasa):
            f = por_clave.get((etiqueta, tasa, ident))
            if f is None:
                return "falta"
            return f"{f['media']:.4f} +- {f['std']:.4f}"
        # La fila uniforme no tiene temperatura: beta = 1/m no pasa por un
        # softmax con tau, asi que imprimir una la haria leer como si corriera
        # a la elegida.
        usada = ("—" if atencion != "learned"
                 else ELEGIDA[etiqueta] if tau is None else tau)
        print(f"{ident:>3} {brazo:>6} {atencion:>9} {str(copia):>6} "
              f"{('on' if loss else 'off'):>5} {str(usada):>9} "
              f"{celda(0.0):>16} {celda(0.4):>16}")
    print(f"azar {1/config.CLASSES:.4f}   techo sin ruido 1.0000")


==== U->M   (filas 3-6 a tau buscada) ====
  #  brazo  atencion  copia  loss       tau    ACC sin ruido    ACC con ruido
  1      B   uniform   None   off         — 0.5278 +- 0.0856 0.5056 +- 0.0930
 2a      B   learned   None   off   buscada 0.5889 +- 0.0692 0.5167 +- 0.0800
 2b      B   learned   None   off       0.5 0.5000 +- 0.0878 0.4611 +- 0.0800
 2c      B   learned   None   off       0.1 0.4944 +- 0.0865 0.3500 +- 0.1050
 3a      E   learned   None   off   buscada 0.0889 +- 0.0534 0.1389 +- 0.0556
 3b      E   learned      7   off   buscada 0.9944 +- 0.0124 0.7222 +- 0.0439
 3c      E   learned      1   off   buscada 0.9944 +- 0.0124 0.7056 +- 0.0465
 3u      E   uniform      7   off         — 0.9944 +- 0.0124 0.8222 +- 0.0847
 4a      E   learned   None    on   buscada 0.1000 +- 0.0152 0.1056 +- 0.0232
 4b      E   learned      7    on   buscada 0.8944 +- 0.1366 0.4778 +- 0.0719
 4c      E   learned      1    on   buscada 0.9333 +- 0.0609 0.6556 +- 0.0751
 5a      F   learned

In [7]:
# El sello: contra qué código corrió esta escalera. Sin él, una corrida vieja y
# una recién hecha se ven idénticas y la vieja se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())

SOURCES-SHA256 357b88aeee6bb2ff76c98e973d83df5b94d1408ac08b03e1ae8df06ab4c90714
